# IMAGE PREPROCESSING
Image preprocesing steps completed in below code.
- Convert to Greyscale (Mode: “L”)
- Noise reduction with Gaussian Blur
- Contrast Enhancement
- Mild sharpening
- Resize image to height to 64 while preserving original width
- Width normalising (Padding or Cropping)
- Pixel Normalisation
- Export as .npy + new manifest generation


In [1]:
import time
NOTEBOOK_START = time.time()

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import cv2
from tqdm.auto import tqdm

IMAGE_TRAIN_DIR = Path("Doctors Handwritten Prescription BD dataset/Training/training_words")
IMAGE_VAL_DIR   = Path("Doctors Handwritten Prescription BD dataset/Validation/validation_words")
IMAGE_TEST_DIR  = Path("Doctors Handwritten Prescription BD dataset/Testing/testing_words")

OUT_DIR = Path("./processed_64x384")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_TRAIN = OUT_DIR / "train"
OUT_VAL   = OUT_DIR / "val"
OUT_TEST  = OUT_DIR / "test"
OUT_TRAIN.mkdir(exist_ok=True)
OUT_VAL.mkdir(exist_ok=True)
OUT_TEST.mkdir(exist_ok=True)

C:\Users\djsyl\anaconda3\envs\torch312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Preprocess function

In [3]:
# preprocess_fast(): helper function (see inline code for details)
def preprocess_fast(img, target_h=64, target_w=384, denoise="gaussian"):
    """
    Robust preprocessing for doctor handwriting crops.
    Accepts:
      - PIL.Image (any mode)
      - numpy array (H,W) or (H,W,C)
    Returns:
      - float32 (target_h, target_w) in [0,1]
    Steps:
      -> grayscale 
      -> denoise 
      -> CLAHE 
      -> sharpen
      -> resize by height (aspect preserved) 
      -> center pad/crop to target_w
      -> normalize
    """
    # ---- 0) Coerce input to a clean uint8 grayscale 2D array ----
    if isinstance(img, Image.Image):
        img = ImageOps.exif_transpose(img)  # important for phone-origin images
        img = img.convert("L")
        img = np.array(img)

    img = np.asarray(img)

    # Handle weird arrays early
    if img.size == 0:
        raise ValueError(f"Empty image array: shape={img.shape}, dtype={img.dtype}")
    if img.dtype == object:
        raise ValueError(f"Object dtype image array (corrupt load): shape={img.shape}")

    # If color slipped in, convert to grayscale
    if img.ndim == 3:
        # best-effort conversion depending on channels
        if img.shape[2] == 3:
            img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        elif img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_RGBA2GRAY)
        else:
            img = img[..., 0]

    if img.ndim != 2:
        raise ValueError(f"Expected 2D grayscale after conversion, got shape={img.shape}")

    # Ensure contiguous + safe dtype/range for OpenCV
    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)
    img = np.ascontiguousarray(img)

    h, w = img.shape
    if h < 2 or w < 2:
        raise ValueError(f"Too-small image: shape={img.shape}")

    # ---- 1) Denoise ----
    try:
        if denoise in ("gaussian", None, "none"):
            # default gaussian unless explicitly disabled
            if denoise == "gaussian":
                img = cv2.GaussianBlur(img, (3, 3), 0)
        if denoise == "median":
            img = cv2.medianBlur(img, 3)
        elif denoise == "bilateral":
            img = cv2.bilateralFilter(img, d=5, sigmaColor=50, sigmaSpace=50)
        # else: no denoise
    except Exception as e:
        raise RuntimeError(f"OpenCV denoise failed: shape={img.shape}, dtype={img.dtype}, err={repr(e)}")

    # ---- 2) Contrast enhancement ----
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)

    # ---- 3) Sharpen ----
    kernel = np.array([[0, -1,  0],
                       [-1,  5, -1],
                       [0, -1,  0]], dtype=np.float32)
    img = cv2.filter2D(img, -1, kernel)

    # ---- 4) Resize by height (preserve aspect) ----
    h, w = img.shape
    scale = target_h / float(h)
    new_w = max(1, int(round(w * scale)))
    img = cv2.resize(img, (new_w, target_h), interpolation=cv2.INTER_AREA)

    # ---- 5) Center pad/crop to target_w ----
    if new_w < target_w:
        pad_total = target_w - new_w
        pad_left = pad_total // 2
        pad_right = pad_total - pad_left
        img = np.pad(img, ((0, 0), (pad_left, pad_right)), mode="constant", constant_values=255)
    else:
        start = (new_w - target_w) // 2
        img = img[:, start:start + target_w]

    # ---- 6) Normalize ----
    return img.astype(np.float32) / 255.0

## Export New .NPY files and manifest function

In [4]:
# export_split(): helper function (see inline code for details)
def export_split(df, image_dir: Path, out_dir: Path,
                 file_col="IMAGE", label_col="MEDICINE_NAME",
                 target_h=64, target_w=384, denoise="gaussian"):
    """
    Exports a processed .npy per image + a manifest.csv.
    Writes manifest_errors.csv with any failures (corrupt/missing images, etc.)
    so one bad file won't crash the whole export.
    """
    image_dir = Path(image_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    records = []
    errors = []

    # tqdm is optional (already imported above)
    iterator = tqdm(df.iterrows(), total=len(df), desc=f"Exporting → {out_dir.name}") if "tqdm" in globals() else df.iterrows()

    for _, row in iterator:
        fname = str(row[file_col])
        label = row[label_col]
        in_path = image_dir / fname

        try:
            if not in_path.exists():
                raise FileNotFoundError(f"Missing file: {in_path}")

            with Image.open(in_path) as im:
                proc = preprocess_fast(im, target_h=target_h, target_w=target_w, denoise=denoise)

            out_path = out_dir / (Path(fname).stem + ".npy")
            np.save(out_path, proc)

            records.append({
                "proc_path": str(out_path),
                "label": label,
                "orig_file": fname,
                "orig_path": str(in_path),
            })

        except Exception as e:
            errors.append({
                "filename": fname,
                "path": str(in_path),
                "error": repr(e),
            })

    manifest = pd.DataFrame(records)
    manifest_path = out_dir / "manifest.csv"
    manifest.to_csv(manifest_path, index=False)

    if errors:
        err_path = out_dir / "manifest_errors.csv"
        pd.DataFrame(errors).to_csv(err_path, index=False)
        print(f"⚠️ Completed with {len(errors)} errors. See: {err_path}")
        print(pd.DataFrame(errors).head(5).to_string(index=False))
    else:
        print("✅ Completed with 0 errors.")

    return manifest_path

In [5]:
fp_train = "Doctors Handwritten Prescription BD dataset/Training/training_labels.csv"
fp_test = "Doctors Handwritten Prescription BD dataset/Testing/testing_labels.csv"
fp_val = "Doctors Handwritten Prescription BD dataset/Validation/validation_labels.csv"

labels_traindf = pd.read_csv(fp_train)
labels_testdf = pd.read_csv(fp_test)
labels_valdf = pd.read_csv(fp_val)

train_manifest = export_split(labels_traindf, IMAGE_TRAIN_DIR, OUT_TRAIN)
val_manifest   = export_split(labels_valdf,  IMAGE_VAL_DIR,   OUT_VAL)
test_manifest  = export_split(labels_testdf, IMAGE_TEST_DIR,  OUT_TEST)

print("Saved manifests:")
print(train_manifest)
print(val_manifest)
print(test_manifest)

Exporting → train: 100%|██████████████████████████████████████████████████████████| 3120/3120 [00:24<00:00, 125.40it/s]


✅ Completed with 0 errors.


Exporting → val: 100%|██████████████████████████████████████████████████████████████| 780/780 [00:06<00:00, 119.15it/s]


✅ Completed with 0 errors.


Exporting → test: 100%|█████████████████████████████████████████████████████████████| 780/780 [00:06<00:00, 119.14it/s]

✅ Completed with 0 errors.
Saved manifests:
processed_64x384\train\manifest.csv
processed_64x384\val\manifest.csv
processed_64x384\test\manifest.csv


In [6]:
elapsed = time.time() - NOTEBOOK_START
print(f"Total notebook runtime: {elapsed/60:.2f} minutes ({elapsed:.1f} seconds)")

Total notebook runtime: 0.64 minutes (38.7 seconds)


## Note: Continues on notebook DL-3-Modelling.ipynb